# Train 3D CNN Classifier

This notebook trains a simple 3D CNN on the zebrafish tensor dataset using **time as channels**. To avoid leakage, it first builds an unaugmented dataset, then performs the train/validation/holdout split, and only then applies random XY rotation augmentation to the training subset.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from zebrafish_ml import (
    Zebrafish3DCNNClassifier,
    augment_training_tensors_with_rotations,
    plot_confusion_matrices,
    plot_training_history,
)
from zebrafish_notebook_utils import (
    configure_full_dataframe_display,
    load_compound_image_condition_map_csv,
)
from zebrafish_tensor_utils import (
    build_moa_labeled_tensor_dataset,
    build_tensor_embedding_2d,
    plot_tensor_embedding_2d,
)

warnings.filterwarnings("ignore", message="IProgress not found.*")

In [ ]:
configure_full_dataframe_display()
condition_df = load_compound_image_condition_map_csv()
condition_df.shape

In [ ]:
# Dataset inputs
selected_mechanisms = [
    "GABAAR_Antagonist",
    "GABAAR_NegativeAllostericModulator",
    "NMDAR_Activation",
    "NMDAR_Antagonist",
]
selected_concentrations = ["high", "mid", "low"]
max_compounds_per_action = 2
max_tensors_per_compound = 8
output_size = (10, 3, 64, 64)
only_active = True
normalize_global_drift = True
loess_frac = 0.25
use_cache = True
use_tiff_cache = True
random_seed = 0

# Split and augmentation inputs
holdout_fraction = 0.25
validation_fraction_within_train = 0.2
train_num_random_rotations = 2
rotation_range_degrees = 5.0

# CNN inputs
conv_channels = (16, 32, 64)
kernel_size_z = (1, 1, 1)
kernel_size_xy = (5, 3, 3)
stride_z = (1, 1, 1)
stride_xy = (1, 1, 1)
pool_kernel_z = (1, 1, 1)
pool_kernel_xy = (2, 2, 2)
pool_stride_z = (1, 1, 1)
pool_stride_xy = (2, 2, 2)
embedding_dim = 64
dropout = 0.2

# Optimizer / training inputs
batch_size = 16
epochs = 20
learning_rate = 1e-3
weight_decay = 1e-4
device = None

# Embedding plot inputs
embedding_method = "umap"
umap_n_neighbors = 15
umap_min_dist = 0.1

In [ ]:
# Build the base dataset without random rotations so we can split first and avoid leakage.
dataset = build_moa_labeled_tensor_dataset(
    condition_df=condition_df,
    selected_mechanisms=selected_mechanisms,
    selected_concentrations=selected_concentrations,
    max_compounds_per_action=max_compounds_per_action,
    max_tensors_per_compound=max_tensors_per_compound,
    output_size=output_size,
    only_active=only_active,
    num_random_rotations=0,
    rotation_range_degrees=rotation_range_degrees,
    normalize_global_drift=normalize_global_drift,
    loess_frac=loess_frac,
    use_cache=use_cache,
    use_tiff_cache=use_tiff_cache,
    random_seed=random_seed,
)

summary_df = pd.DataFrame(
    [
        {"metric": "n_examples", "value": int(dataset["tensors"].shape[0])},
        {"metric": "tensor_shape", "value": tuple(dataset["tensors"].shape)},
        {"metric": "n_classes_including_water", "value": int(len(dataset["label_map"]))},
    ]
)
display(summary_df)
display(dataset["metadata"].groupby(["label", "label_name"]).size().reset_index(name="n_examples"))

In [ ]:
all_indices = np.arange(len(dataset["labels"]))
all_labels = dataset["labels"].detach().cpu().numpy()

train_val_indices, holdout_indices = train_test_split(
    all_indices,
    test_size=holdout_fraction,
    random_state=random_seed,
    stratify=all_labels,
)
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=validation_fraction_within_train,
    random_state=random_seed,
    stratify=all_labels[train_val_indices],
)

X_train_base = dataset["tensors"][train_indices]
y_train_base = dataset["labels"][train_indices]
metadata_train_base = dataset["metadata"].iloc[train_indices].reset_index(drop=True)

X_val = dataset["tensors"][val_indices]
y_val = dataset["labels"][val_indices].detach().cpu().numpy()
metadata_val = dataset["metadata"].iloc[val_indices].reset_index(drop=True)

X_holdout = dataset["tensors"][holdout_indices]
y_holdout = dataset["labels"][holdout_indices].detach().cpu().numpy()
metadata_holdout = dataset["metadata"].iloc[holdout_indices].reset_index(drop=True)

X_train, y_train, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    y_train_base,
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=random_seed,
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train_base", "n_examples": int(len(X_train_base))},
        {"split": "train_augmented", "n_examples": int(len(X_train))},
        {"split": "val", "n_examples": int(len(X_val))},
        {"split": "holdout", "n_examples": int(len(X_holdout))},
    ]
)
display(split_summary_df)

In [ ]:
model = Zebrafish3DCNNClassifier(
    conv_channels=conv_channels,
    kernel_size_z=kernel_size_z,
    kernel_size_xy=kernel_size_xy,
    stride_z=stride_z,
    stride_xy=stride_xy,
    pool_kernel_z=pool_kernel_z,
    pool_kernel_xy=pool_kernel_xy,
    pool_stride_z=pool_stride_z,
    pool_stride_xy=pool_stride_xy,
    embedding_dim=embedding_dim,
    dropout=dropout,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    validation_split=0.0,
    random_state=random_seed,
    device=device,
    verbose=True,
)

model.fit(X_train, y_train, validation_data=(X_val, y_val))
model.history_

In [ ]:
plot_training_history(model, title="3D CNN train/validation loss");

In [ ]:
holdout_pred = model.predict(X_holdout)
report_df = pd.DataFrame(
    classification_report(
        y_holdout,
        holdout_pred,
        labels=model.classes_,
        target_names=[dataset["label_map"][int(label)] for label in model.classes_],
        output_dict=True,
        zero_division=0,
    )
).T
display(report_df)
plot_confusion_matrices(
    y_holdout,
    holdout_pred,
    class_labels=model.classes_,
    label_map=dataset["label_map"],
);

In [ ]:
holdout_embeddings = torch.from_numpy(model.transform(X_holdout))
holdout_embedding_df = build_tensor_embedding_2d(
    holdout_embeddings,
    y_holdout,
    label_map=dataset["label_map"],
    metadata=metadata_holdout,
    method=embedding_method,
    random_state=random_seed,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
)
display(holdout_embedding_df.head(5))
plot_tensor_embedding_2d(
    holdout_embedding_df,
    title=f"Holdout learned embeddings | {embedding_method.upper()}",
    marker_column="compound",
);